# Stage 2: 4-way role classification

Classifies among the 4 substantive legal roles only (no `None`) — trained and evaluated on gold-relevant sentences, i.e. relevance is assumed already known.

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEST_DF_PATH = Path("predictions/test_df.csv")
REGISTRY_PATH = Path("roberta_models/best_models_registry.json")
MODELS_DIR = Path("roberta_models")

TEXT_COL = "sent_text"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LEGAL_LABELS = ["beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]

print("Device:", DEVICE)

Device: cuda


## 2. Load test data

Inference runs over all 1502 test sentences (so this notebook's output is reusable elsewhere, e.g. by pipeline notebooks), but evaluation is restricted to the 961 gold-relevant sentences, matching how the model was trained.

In [2]:
test_df = pd.read_csv(TEST_DF_PATH, keep_default_na=False)

print("Test rows:", len(test_df))
print("Gold-relevant rows for evaluation:", test_df["label"].isin(LEGAL_LABELS).sum())
test_df["label"].value_counts()

Test rows: 1502
Gold-relevant rows for evaluation: 961


label
None                 541
beoordeling          389
materiele feiten     297
proceshandelingen    206
beslissing            69
Name: count, dtype: int64

## 3. Load the Stage 2 model

In [3]:
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)

model_info = registry["stage2_text_only_4way"]
model_path = MODELS_DIR / Path(model_info["saved_path"]).name
labels = model_info["labels_order"]
max_len = model_info.get("max_len", 256)

if not model_path.exists():
    raise FileNotFoundError(f"Model path not found: {model_path}")

print("Model path:", model_path)
print("Labels (in order):", labels)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(DEVICE)
model.eval()

Model path: roberta_models\stage2_text_only_4way_best_20260203_105457
Labels (in order): ['beoordeling', 'beslissing', 'materiele feiten', 'proceshandelingen']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(40000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

## 4. Run inference

In [4]:
@torch.no_grad()
def predict_probs(texts, tokenizer, model, max_len, batch_size=32):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Stage 2 inference"):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)


texts = test_df[TEXT_COL].fillna("").astype(str).tolist()
probs = predict_probs(texts, tokenizer, model, max_len, BATCH_SIZE)

pred_idx = probs.argmax(axis=1)
pred_label = np.array([labels[i] for i in pred_idx])

probs.shape

Stage 2 inference:   0%|          | 0/47 [00:00<?, ?it/s]

(1502, 4)

## 5. Attach predictions and save

In [5]:
for i, label in enumerate(labels):
    col = "stage2_p_" + label.lower().replace(" ", "_")
    test_df[col] = probs[:, i]

test_df["stage2_pred_idx"] = pred_idx
test_df["stage2_pred_label"] = pred_label

OUTPUT_PATH = Path("predictions/stage2_fourway_inference.csv")
test_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)
test_df[["sent_text", "label", "stage2_pred_label"]].head()

Saved: predictions\stage2_fourway_inference.csv


,sent_text,label,stage2_pred_label
0,De kantonrechter heeft in het bestreden vonnis...,None,beoordeling
1,Deze feiten zijn in hoger beroep niet in gesch...,None,beoordeling
2,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,materiele feiten
3,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,materiele feiten
4,Deze verklaring houdt onder meer in: “Dit bedr...,None,proceshandelingen


## 6. Evaluate

Only on gold-relevant sentences. Sanity check: should reproduce the previously reported Stage 2 numbers (accuracy 0.7315, macro F1 0.7526).

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()
print("Rows used for evaluation:", len(legal_df))

y_true = legal_df["label"]
y_pred = legal_df["stage2_pred_label"]

acc = accuracy_score(y_true, y_pred)
print("\nStage 2 accuracy:", round(acc, 4))

print("\nClassification Report")
print(
    classification_report(
        y_true, y_pred, labels=LEGAL_LABELS, digits=4, zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred, labels=LEGAL_LABELS)
pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in LEGAL_LABELS],
    columns=[f"pred_{l}" for l in LEGAL_LABELS]
)

Rows used for evaluation: 961

Stage 2 accuracy: 0.7315

Classification Report
                   precision    recall  f1-score   support

      beoordeling     0.7364    0.8329    0.7817       389
       beslissing     0.9677    0.8696    0.9160        69
 materiele feiten     0.7720    0.6498    0.7057       297
proceshandelingen     0.6029    0.6117    0.6072       206

         accuracy                         0.7315       961
        macro avg     0.7697    0.7410    0.7526       961
     weighted avg     0.7354    0.7315    0.7304       961



,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_beoordeling,324,2,36,27
true_beslissing,7,60,2,0
true_materiele feiten,48,0,193,56
true_proceshandelingen,61,0,19,126


## 7. Header-prior Bayesian fusion

Same methodology as `01_five_way_inference.ipynb` section 8. The header prior here is built only from the legal-labelled subset of the training set, matching how Stage 2 itself was trained (it never saw `None` examples).

In [7]:
from sklearn.metrics import precision_recall_fscore_support

TRAIN_DF_PATH = Path("predictions/train_df.csv")
EVAL_DF_PATH = Path("predictions/eval_df.csv")
ALPHA = 1.0

train_df = pd.read_csv(TRAIN_DF_PATH, keep_default_na=False)
eval_df = pd.read_csv(EVAL_DF_PATH, keep_default_na=False)
train_legal = train_df[train_df["label"].isin(LEGAL_LABELS)].copy()

print("Train rows (legal-labelled only):", len(train_legal))
print("Validation rows:", len(eval_df))


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    counts = (
        df_train.groupby(["hdr_group", label_col]).size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    probs = counts + alpha
    return probs.div(probs.sum(axis=1), axis=0)


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    return (counts / counts.sum()).values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    def get_prior(hdr_group):
        if hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    mat = np.vstack(df_apply["hdr_group"].apply(get_prior))
    return pd.DataFrame(mat, columns=labels, index=df_apply.index)


def combine_log_scores(text_probs, metadata_probs, lam, eps=1e-12):
    text = np.clip(text_probs.values, eps, 1.0)
    metadata = np.clip(metadata_probs.values, eps, 1.0)

    scores = np.log(text) + lam * np.log(metadata)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    scores /= scores.sum(axis=1, keepdims=True)

    return pd.DataFrame(scores, columns=text_probs.columns, index=text_probs.index)

Train rows (legal-labelled only): 3611
Validation rows: 1281


In [8]:
eval_texts = eval_df[TEXT_COL].fillna("").astype(str).tolist()
eval_probs = predict_probs(eval_texts, tokenizer, model, max_len, BATCH_SIZE)
eval_text_probs_full = pd.DataFrame(eval_probs, columns=labels, index=eval_df.index)[LEGAL_LABELS]

eval_legal = eval_df[eval_df["label"].isin(LEGAL_LABELS)].copy()
eval_text_probs = eval_text_probs_full.loc[eval_legal.index]

header_prior_s2 = make_header_prior(train_legal, "label", LEGAL_LABELS, ALPHA)
global_prior_s2 = make_global_prior(train_legal, "label", LEGAL_LABELS, ALPHA)
eval_meta_prior = get_meta_prior_df(eval_legal, header_prior_s2, global_prior_s2, LEGAL_LABELS)

LAMBDA_GRID = [0, 0.1, 0.25, 0.5, 1, 2, 3, 5]
sweep_results = []

for lam in LAMBDA_GRID:
    fused = combine_log_scores(eval_text_probs, eval_meta_prior, lam)
    pred = fused.idxmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        eval_legal["label"], pred, labels=LEGAL_LABELS, average="macro", zero_division=0
    )
    sweep_results.append({"lambda": lam, "val_macro_f1": round(f1, 4)})

sweep_df = pd.DataFrame(sweep_results)
best_lambda = sweep_df.loc[sweep_df["val_macro_f1"].idxmax(), "lambda"]

print("Validation lambda sweep (evaluated on gold-relevant validation sentences):")
print(sweep_df)
print(f"\nBest lambda on validation: {best_lambda}")

Stage 2 inference:   0%|          | 0/41 [00:00<?, ?it/s]

Validation lambda sweep (evaluated on gold-relevant validation sentences):
   lambda  val_macro_f1
0    0.00        0.7827
1    0.10        0.7850
2    0.25        0.7914
3    0.50        0.7960
4    1.00        0.7717
5    2.00        0.7642
6    3.00        0.7690
7    5.00        0.7138

Best lambda on validation: 0.5


In [9]:
test_prob_cols = {l: "stage2_p_" + l.lower().replace(" ", "_") for l in LEGAL_LABELS}
legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()

test_text_probs = legal_df[[test_prob_cols[l] for l in LEGAL_LABELS]].copy()
test_text_probs.columns = LEGAL_LABELS

test_meta_prior = get_meta_prior_df(legal_df, header_prior_s2, global_prior_s2, LEGAL_LABELS)

results = []

for lam, name in [(0, "no fusion (lambda=0)"), (1, "lambda = 1"), (best_lambda, f"tuned lambda ({best_lambda})")]:
    fused_test = combine_log_scores(test_text_probs, test_meta_prior, lam)
    pred_test = fused_test.idxmax(axis=1)

    acc = accuracy_score(legal_df["label"], pred_test)
    _, _, f1, _ = precision_recall_fscore_support(
        legal_df["label"], pred_test, labels=LEGAL_LABELS, average="macro", zero_division=0
    )
    results.append({"setting": name, "lambda": lam, "test_accuracy": round(acc, 4), "test_macro_f1": round(f1, 4)})

results_df = pd.DataFrame(results)
print(results_df)

                setting  lambda  test_accuracy  test_macro_f1
0  no fusion (lambda=0)     0.0         0.7315         0.7526
1            lambda = 1     1.0         0.7638         0.7779
2    tuned lambda (0.5)     0.5         0.7492         0.7687


In [10]:
print(f"Full classification report at tuned lambda={best_lambda} (test set)")
print(classification_report(legal_df["label"], pred_test, labels=LEGAL_LABELS, digits=4, zero_division=0))

Full classification report at tuned lambda=0.5 (test set)
                   precision    recall  f1-score   support

      beoordeling     0.7438    0.8509    0.7938       389
       beslissing     0.9836    0.8696    0.9231        69
 materiele feiten     0.7976    0.6768    0.7322       297
proceshandelingen     0.6305    0.6214    0.6259       206

         accuracy                         0.7492       961
        macro avg     0.7889    0.7546    0.7687       961
     weighted avg     0.7534    0.7492    0.7481       961

